# Final 30×30 experiment

This notebook only loads the persisted CSV results from `run_query_experiment.py`. It does not import or rerun the optimizer. Each method/objective pair was run `NUM_RUNS` times with different optimizer seeds; every run's trace is interpolated onto a common query grid, and each figure shows the median across seeds with a shaded interquartile band.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_experiment_directory():
    cwd = Path.cwd().resolve()
    if (cwd / 'data' / 'objective_vs_queries.csv').exists():
        return cwd
    candidate = cwd / 'experiments' / 'final_experiments'
    if (candidate / 'data' / 'objective_vs_queries.csv').exists():
        return candidate
    raise FileNotFoundError('Run run_query_experiment.py before plotting')


EXPERIMENT_DIR = find_experiment_directory()
DATA_DIR = EXPERIMENT_DIR / 'data'
FIGURE_DIR = EXPERIMENT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

query_results = pd.read_csv(DATA_DIR / 'objective_vs_queries.csv')

OBJECTIVES = ('ky_fan_regression', 'singular_value_sum')
EXPECTED_ALGORITHMS = {
    'adjoint_sensing_operator',
    'alternating_projections',
    'burer_monteiro_gradient_descent',
    'IHT_SpecGD',
    'iterative_hard_thresholding',
    'lozo',
    'pseudoinverse',
}
NUM_RUNS = 10
MAX_QUERIES = 100_000

successes = query_results[query_results.status == 'success']
assert set(successes.algorithm) == EXPECTED_ALGORITHMS
assert set(successes.objective) == set(OBJECTIVES)

# Every method/objective pair must have all NUM_RUNS seeds present.
seed_counts = successes.groupby(['algorithm', 'objective']).random_seed.nunique()
incomplete = seed_counts[seed_counts != NUM_RUNS]
assert incomplete.empty, f'Pairs missing seeds:\n{incomplete}'

failures = query_results[query_results.status == 'failed']
assert failures.empty, f'Failed runs present:\n{failures[["algorithm", "objective", "random_seed", "error_message"]]}'

print(f'{len(successes)} trace points across {len(seed_counts)} method/objective pairs, {NUM_RUNS} seeds each')

In [ ]:
# Objective value versus cumulative query count: median across seeds with IQR band.
QUERY_GRID = np.linspace(0, MAX_QUERIES, 500)

for objective in OBJECTIVES:
    figure, axis = plt.subplots(figsize=(11, 7))
    objective_data = successes[successes.objective == objective]

    for algorithm, algorithm_data in objective_data.groupby('algorithm', sort=True):
        # Interpolate each seed's trace onto the common query grid. np.interp
        # clamps at the trace ends, so a run that stopped just short of
        # MAX_QUERIES holds its final value for the last grid points.
        interpolated_runs = []
        for _, run_data in algorithm_data.groupby('random_seed'):
            run_data = run_data.sort_values('queries')
            interpolated_runs.append(
                np.interp(QUERY_GRID, run_data.queries, run_data.objective_value)
            )
        interpolated_runs = np.vstack(interpolated_runs)

        median = np.median(interpolated_runs, axis=0)
        lower_quartile = np.percentile(interpolated_runs, 25, axis=0)
        upper_quartile = np.percentile(interpolated_runs, 75, axis=0)

        (line,) = axis.plot(
            QUERY_GRID,
            median,
            linewidth=1.5,
            label=algorithm.replace('_', ' '),
        )
        axis.fill_between(
            QUERY_GRID,
            lower_quartile,
            upper_quartile,
            color=line.get_color(),
            alpha=0.2,
            linewidth=0,
        )

    axis.set_xlabel('Cumulative number of queries', fontsize=18)
    axis.set_ylabel('Objective-function value', fontsize=18)
    axis.set_yscale('symlog', linthresh=1e-12)
    axis.grid(True, which='both', alpha=0.25)
    if objective == 'singular_value_sum':
        axis.legend(fontsize=12)
    figure.tight_layout()

    output_path = FIGURE_DIR / f'objective_vs_queries_{objective}.png'
    figure.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved {output_path}')